# Tutorial A03: Shapefile Indexing, WKT Extraction and Attribute Query

This notebook demonstrates how to:

1. **Load and inspect** a Shapefile using GeoPandas
2. **Extract all feature geometries** as a Python list of WKT strings
3. **Query features by attribute** (single condition and multi-condition)
4. **Export selected WKT** for use in `s1grits` config (ROI definition)
5. **Save a filtered subset** as a new Shapefile

---

## How to Get a WKT String

There are three ways to get a WKT polygon for your area of interest:

**1. ASF Vertex web portal (fastest, no tools needed)**

1. Go to https://search.asf.alaska.edu
2. Click the polygon or rectangle draw tool in the top-right toolbar
3. Draw your area of interest on the map
4. The WKT string appears automatically in the search bar — copy it directly

This WKT can be pasted straight into your `s1grits_config_base_en.yaml` under `roi.wkt`.

**2. geojson.io (draw and export)**

1. Go to https://geojson.io
2. Draw a polygon using the toolbar on the right
3. In the JSON panel on the right, copy the coordinates
4. Convert to WKT format: `POLYGON((lon1 lat1, lon2 lat2, ...))`

**3. From a Shapefile (this notebook)**

Use this notebook when you need to derive a precise ROI from an existing administrative boundary or custom polygon dataset.

---

## Prerequisites

```bash
conda activate py312_s1grits
```

| Package | Purpose |
| :--- | :--- |
| `geopandas` | Read / write Shapefiles and GeoJSON |
| `shapely` | WKT serialization and geometry utilities |
| `pandas` | Tabular attribute inspection |

**Where to get Shapefiles:**

- GADM administrative boundaries: https://gadm.org
- Natural Earth (countries, states, coastlines): https://www.naturalearthdata.com
- Your own field survey or GIS export


## 0. Imports


In [ ]:
import geopandas as gpd
import pandas as pd
from pathlib import Path
from shapely.wkt import dumps as wkt_dumps

print(f"INFO  GeoPandas version : {gpd.__version__}")
print(f"INFO  Pandas   version  : {pd.__version__}")


## 1. Configuration

Set `SHP_PATH` to your Shapefile before running the cells below.

The examples use a GADM administrative boundary file, but any polygon or point Shapefile works.
GeoPandas also reads `.geojson` and `.gpkg` files — all OGR-supported formats are accepted.

> **Note**: You must set `SHP_PATH` to a valid file on your system before running the assertion below.
> Download shapefiles from GADM (https://gadm.org) or Natural Earth (https://www.naturalearthdata.com).


In [ ]:
# ---- USER CONFIGURATION -----------------------------------------------
# Set this to your local Shapefile path before running.
# Download from GADM (https://gadm.org), Natural Earth, or use your own.
SHP_PATH = Path(r"YOUR_SHAPEFILE_PATH")   # e.g. Path(r"D:/your/data/boundary.shp")

# Output path for the filtered Shapefile (used in Section 7)
OUT_SHP = Path(r"YOUR_OUTPUT_SHAPEFILE_PATH")  # e.g. Path(r"D:/your/data/filtered_output.shp")
# -----------------------------------------------------------------------

try:
    assert SHP_PATH.exists(), f"ERROR  Shapefile not found: {SHP_PATH}"
    print(f"INFO  Loading: {SHP_PATH}")
except AssertionError as e:
    print(e)
    print("INFO  Please set SHP_PATH to a valid Shapefile path above and re-run this cell.")


## 2. Load the Shapefile and Inspect its Structure


In [ ]:
gdf = gpd.read_file(SHP_PATH)

print(f"INFO  Total features : {len(gdf)}")
print(f"INFO  CRS            : {gdf.crs}")
print(f"INFO  Geometry type  : {gdf.geom_type.unique().tolist()}")
print(f"INFO  Columns        : {gdf.columns.tolist()}")
print()

# Show first few rows (attribute columns only, no geometry)
attr_cols = [c for c in gdf.columns if c != 'geometry']
display(gdf[attr_cols].head(10))


## 3. Extract ALL Feature Geometries as a WKT List

The simplest way to get all WKT strings is via the `geometry` column's built-in serializer.
Each element in `wkt_list` corresponds to one feature row.


In [ ]:
# Method 1: via GeoSeries.to_wkt() (fastest, recommended)
wkt_list = gdf.geometry.to_wkt().tolist()

print(f"INFO  Total WKT entries : {len(wkt_list)}")
print()

# Preview the first 5 entries (truncated to 120 characters each)
for i, wkt in enumerate(wkt_list[:5]):
    preview = wkt[:120] + ("..." if len(wkt) > 120 else "")
    print(f"  [{i:3d}]  {preview}")


In [ ]:
# Method 2: paired name + WKT (useful when you need both)
# Adjust 'NAME_1' to whatever the name column is in your Shapefile.
name_col = "NAME_1"   # <-- change if needed

if name_col in gdf.columns:
    name_wkt_pairs = list(zip(gdf[name_col], gdf.geometry.to_wkt()))
    print(f"INFO  Name-WKT pairs (first 5):")
    for name, wkt in name_wkt_pairs[:5]:
        preview = wkt[:100] + "..."
        print(f"  {name:<30s}  {preview}")
else:
    print(f"WARN  Column '{name_col}' not found. Available columns: {gdf.columns.tolist()}")


## 4. Single-Condition Attribute Query

Filter features by matching a specific column value and return the WKT list.


In [ ]:
# ---- USER CONFIGURATION -----------------------------------------------
QUERY_COL   = "NAME_1"       # Column to filter on
QUERY_VALUE = "Bayern"       # Value to match (exact, case-sensitive)
# -----------------------------------------------------------------------

if QUERY_COL not in gdf.columns:
    print(f"ERROR  Column '{QUERY_COL}' not found. Available: {gdf.columns.tolist()}")
else:
    selected = gdf[gdf[QUERY_COL] == QUERY_VALUE]

    if len(selected) == 0:
        unique_vals = sorted(gdf[QUERY_COL].dropna().unique().tolist())
        print(f"WARN  No features matched '{QUERY_VALUE}'.")
        print(f"      Unique values in '{QUERY_COL}':")
        for v in unique_vals:
            print(f"        - {v}")
    else:
        selected_wkt_list = selected.geometry.to_wkt().tolist()
        print(f"INFO  Matched {len(selected_wkt_list)} feature(s) where {QUERY_COL} == '{QUERY_VALUE}'")
        for i, wkt in enumerate(selected_wkt_list):
            preview = wkt[:120] + ("..." if len(wkt) > 120 else "")
            print(f"  [{i}]  {preview}")


## 5. Multi-Condition Query

Combine multiple filters using `&` (AND) or `|` (OR).
Always wrap each condition in parentheses.


In [ ]:
# ---- USER CONFIGURATION -----------------------------------------------
# Example: select features where TYPE_1 == 'Freistaat' AND NAME_1 starts with 'B'
COL_A   = "TYPE_1"
VALUE_A = "Freistaat"

COL_B   = "NAME_1"
VALUE_B = "B"    # prefix match via .str.startswith()
# -----------------------------------------------------------------------

missing = [c for c in [COL_A, COL_B] if c not in gdf.columns]
if missing:
    print(f"WARN  Columns not found: {missing}. Skipping multi-condition query.")
    print(f"      Available columns: {gdf.columns.tolist()}")
else:
    mask = (gdf[COL_A] == VALUE_A) & (gdf[COL_B].str.startswith(VALUE_B))
    multi_selected = gdf[mask]

    print(f"INFO  Multi-condition query matched {len(multi_selected)} feature(s):")
    attr_cols = [c for c in multi_selected.columns if c != 'geometry']
    display(multi_selected[attr_cols])

    multi_wkt_list = multi_selected.geometry.to_wkt().tolist()
    print(f"\nINFO  WKT list ({len(multi_wkt_list)} entries):")
    for i, wkt in enumerate(multi_wkt_list):
        preview = wkt[:120] + ("..." if len(wkt) > 120 else "")
        print(f"  [{i}]  {preview}")


## 6. Export Selected WKT for s1grits Config

When a single feature is selected, copy the WKT into your `s1grits_config_base_en.yaml` under `roi.wkt`.
When multiple features are selected, dissolve them first to get a unified polygon.


In [ ]:
# ---- USER CONFIGURATION -----------------------------------------------
# Set this to the GeoDataFrame you want to export as ROI WKT.
# Options: 'selected' (single-condition result) or 'multi_selected'
roi_gdf = selected   # <-- change if needed
# -----------------------------------------------------------------------

if 'roi_gdf' not in dir() or len(roi_gdf) == 0:
    print("WARN  roi_gdf is empty or not defined. Run a query cell above first.")
else:
    if len(roi_gdf) == 1:
        # Single feature: use geometry directly
        roi_wkt = roi_gdf.geometry.iloc[0].wkt
        print("INFO  Single feature selected -- using geometry directly.")
    else:
        # Multiple features: dissolve into one unified polygon
        dissolved = roi_gdf.dissolve()
        roi_wkt = dissolved.geometry.iloc[0].wkt
        print(f"INFO  {len(roi_gdf)} features dissolved into one polygon.")

    print()
    print("=" * 70)
    print("Copy the WKT below into your s1grits_config_base_en.yaml:")
    print("=" * 70)
    print()
    print("roi:")
    print(f'  wkt: "{roi_wkt}"')
    print()
    print("=" * 70)


## 7. Save Filtered Result as a New Shapefile (Optional)

Persist the filtered GeoDataFrame as a standalone Shapefile for use in QGIS or other tools.


In [ ]:
# OUT_SHP is defined in the Configuration cell (Section 1)
save_gdf = selected   # GeoDataFrame to save; change to multi_selected if needed

if 'save_gdf' not in dir() or len(save_gdf) == 0:
    print("WARN  save_gdf is empty or not defined. Run a query cell above first.")
else:
    OUT_SHP.parent.mkdir(parents=True, exist_ok=True)
    save_gdf.to_file(OUT_SHP)
    print(f"INFO  Saved {len(save_gdf)} feature(s) to: {OUT_SHP}")


---

## Summary

| Task | Code pattern |
| :--- | :--- |
| Load Shapefile | `gdf = gpd.read_file(path)` |
| All WKT as list | `gdf.geometry.to_wkt().tolist()` |
| Single-condition filter | `gdf[gdf['COL'] == value]` |
| Multi-condition filter | `gdf[(gdf['A'] == x) & (gdf['B'] == y)]` |
| Dissolve to one polygon | `gdf.dissolve().geometry.iloc[0].wkt` |
| Save filtered result | `gdf.to_file(path)` |

The WKT string produced in **Section 6** can be pasted directly into `s1grits_config_base_en.yaml` under `roi.wkt` to define the processing region for `s1grits process`.
